In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import json
from pathlib import Path

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from utils.tools import aggregate_results

In [3]:
def get_suite(row):

    n_demos = row["number of demonstrations"]
    type_demos = row["type of demonstrations"][0:3]
    instr = "impl" if row["use instructions"] == "no" else "expl"

    return f"{n_demos}-{type_demos}-{instr}"

def capitalize(s):
    return s[0].upper() + s[1:]

def order_suites(strings):
    return sorted(strings, key=lambda s: (
        s.split('-')[2],  # implicit-excplict
        s.split('-')[0],  # num demos
        s.split('-')[1],  # demo type
    ))

def order_models(strings):
    return sorted(strings, key=lambda s: (
        int(s.split('-')[1][:-1]),  # Param count, B dropped
        #s.split('-')[0],  # Model name
    ), reverse=True)

def bold_extreme_values(s, by_model=True):
    # Bold max for mean

    if not by_model:
        is_max = s == s.max()
        return ['font-weight: bold' if v else '' for v in is_max]
    
    font_array = []

    model_level = s.index.names.index('model')
    models = s.index.get_level_values(model_level)
    models = pd.Series(list(models)).unique()

    idx = pd.IndexSlice
    
    for model in models:   
        values_by_model = s.loc[idx[model]]
        is_max = values_by_model == values_by_model.max()
        font_array += ['font-weight: bold' if v else '' for v in is_max]

    return font_array

In [4]:
res = pd.read_csv("./data/metrics.csv", sep=";")
#res = res.set_index(['model', 'number of demonstrations', 'type of demonstrations', 'use instructions'])
#res = res.sort_values(by=["use instructions", "number of demonstrations", "type of demonstrations"])

res["suite"] = res.apply(get_suite, axis=1)
res = res.set_index(["model", "suite"])

In [5]:
std_cols = [c for c in res.columns if "std" in c and "total" not in c]
std_df = res[std_cols].copy()

parts = std_df.columns.str.split(' ')

std_df.columns = pd.MultiIndex.from_tuples(
    [(p[0], p[1]) for p in parts],
    names=["label", "metric"]
)

pd.options.display.max_columns = None
pd.options.display.max_rows = None

print(std_df.style.apply(bold_extreme_values).to_latex(convert_css=True))

\begin{tabular}{llrrrrrrrrrrrr}
 & label & \multicolumn{4}{r}{theme} & \multicolumn{4}{r}{topic} & \multicolumn{4}{r}{concept} \\
 & metric & accuracy & precision & recall & f1 & accuracy & precision & recall & f1 & accuracy & precision & recall & f1 \\
model & suite &  &  &  &  &  &  &  &  &  &  &  &  \\
\multirow[c]{11}{*}{Llama-8B} & 0-non-expl & 0.005050 & 0.004933 & 0.001922 & 0.003427 & 0.006255 & 0.009577 & 0.007709 & 0.007386 & 0.008627 & 0.006507 & 0.006995 & 0.006154 \\
 & 1-neg-impl & 0.050409 & \bfseries 0.082297 & \bfseries 0.158272 & \bfseries 0.144414 & 0.045867 & \bfseries 0.079976 & \bfseries 0.157214 & \bfseries 0.135791 & 0.044579 & 0.169873 & \bfseries 0.210885 & 0.056043 \\
 & 1-neg-expl & \bfseries 0.060690 & 0.037805 & 0.122982 & 0.110451 & 0.035447 & 0.040069 & 0.074051 & 0.115696 & 0.049023 & 0.037475 & 0.017312 & 0.023864 \\
 & 1-pos-impl & 0.038072 & 0.043300 & 0.011215 & 0.035564 & 0.038017 & 0.048754 & 0.000000 & 0.038979 & 0.049261 & \bfseries 0.174165 & 0

In [6]:
score_cols = [c for c in res.columns if "mean" in c and "total" not in c]
means = res[score_cols].copy()

means.columns = [" ".join([capitalize(p) for p in col.split(" ")]) for col in means.columns]

parts = means.columns.str.split(' ')
means.columns = pd.MultiIndex.from_tuples(
    [(p[0], p[1]) for p in parts],
    names=["Label", "Metric"]
)

print(means.style.apply(bold_extreme_values).to_latex(convert_css=True))

\begin{tabular}{llrrrrrrrrrrrr}
 & Label & \multicolumn{4}{r}{Theme} & \multicolumn{4}{r}{Topic} & \multicolumn{4}{r}{Concept} \\
 & Metric & Accuracy & Precision & Recall & F1 & Accuracy & Precision & Recall & F1 & Accuracy & Precision & Recall & F1 \\
model & suite &  &  &  &  &  &  &  &  &  &  &  &  \\
\multirow[c]{11}{*}{Llama-8B} & 0-non-expl & 0.770841 & 0.713261 & 0.952982 & 0.815870 & 0.807103 & 0.928683 & 0.667658 & 0.776810 & 0.683364 & 0.632032 & 0.919273 & 0.749048 \\
 & 1-neg-impl & 0.603075 & 0.815483 & 0.349357 & 0.467009 & 0.660846 & 0.841676 & 0.424158 & 0.542374 & 0.711818 & 0.806681 & 0.679131 & 0.699036 \\
 & 1-neg-expl & 0.663670 & 0.874339 & 0.431579 & 0.570127 & 0.547566 & 0.927327 & 0.109294 & 0.189303 & 0.650562 & 0.600654 & \bfseries 0.970803 & 0.741252 \\
 & 1-pos-impl & 0.627136 & 0.572761 & 0.961074 & 0.717053 & 0.685372 & 0.592492 & \bfseries 1.000000 & 0.743152 & 0.658201 & 0.811988 & 0.553712 & 0.635414 \\
 & 1-pos-expl & 0.766292 & 0.797598 & 0.757746 &